# Perceptron bonus tasks

This notebook contains:
- **Kozinec’s algorithm** (binary linear discriminant, same feasibility problem as perceptron).
- **Multi-class perceptron** training algorithm described in the assignment (mean-init + misclassified-sample updates).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# If you want reproducible plots
np.random.seed(0)


In [ ]:
def _make_augmented(X):
    """Add bias feature: returns Z = [X; 1]. X is (d, N)."""
    X = np.asarray(X)
    return np.vstack([X, np.ones((1, X.shape[1]))])

def _labels_to_sign(y):
    """Map labels to signs s in {+1,-1}. Accepts y in {0,1} or {-1,+1}."""
    y = np.asarray(y).ravel()
    uniq = np.unique(y)
    if np.all(np.isin(uniq, [0, 1])):
        return 1 - 2*y
    return y.astype(float)

def plot_binary_boundary_2d(X, y, w, b, title=None):
    """Plot 2D points and linear boundary w^T x + b = 0 (labels assumed 0/1 or -1/+1)."""
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    plt.figure()
    if set(np.unique(y)).issubset({-1, 1}):
        y01 = (y < 0).astype(int)  # -1 -> 1, +1 -> 0 for coloring
    else:
        y01 = y

    plt.scatter(X[0, y01 == 0], X[1, y01 == 0], s=25, label="class 0")
    plt.scatter(X[0, y01 == 1], X[1, y01 == 1], s=25, label="class 1")

    x_min, x_max = X[0].min() - 1, X[0].max() + 1
    xs = np.linspace(x_min, x_max, 200)

    if abs(w[1]) > 1e-12:
        ys = -(w[0]*xs + b) / w[1]
        plt.plot(xs, ys)
    else:
        # vertical line: w0*x + b = 0
        x0 = -b / w[0] if abs(w[0]) > 1e-12 else 0.0
        plt.axvline(x0)

    plt.legend()
    if title:
        plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()


## 1) Kozinec’s algorithm (binary)

We work with the same feasibility system as the perceptron:

- Build transformed samples `z_i = s_i * [x_i; 1]` so that we want `alpha^T z_i > 0` for all `i`.
- Start with `alpha_1` as any point in the convex hull (we pick one sample).
- Repeatedly find a violated sample `x_t` with `alpha^T x_t <= 0`.
- Update `alpha` by moving on the line segment between `alpha` and `x_t` to the point closest to the origin:

\[
\alpha_{t+1} = (1-k)\alpha_t + k x_t, \quad
k = \frac{\alpha_t^T(\alpha_t - x_t)}{\|x_t - \alpha_t\|^2}
\]

We clamp `k` to `[0, 1]` to stay on the segment (convex-hull form).


In [ ]:
def kozinec(X, y, max_iterations=10000):
    """Kozinec's algorithm. Returns (w, b) or (nan, nan) if not found.

    X: (d, N)
    y: (N,) in {0,1} or {-1,+1}
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    N = X.shape[1]
    d = X.shape[0]

    Z = _make_augmented(X)          # (d+1, N)
    s = _labels_to_sign(y)          # (N,)
    Zt = Z * s                      # (d+1, N): want alpha^T Zt[:,i] > 0

    alpha = Zt[:, 0].astype(float).copy()

    for _ in range(int(max_iterations)):
        margins = alpha @ Zt
        bad = np.flatnonzero(margins <= 0)
        if bad.size == 0:
            return alpha[:-1].copy(), float(alpha[-1])

        xt = Zt[:, int(bad[0])]

        dvec = xt - alpha
        denom = float(dvec @ dvec)
        if denom <= 1e-18:
            continue

        k = float((alpha @ (alpha - xt)) / denom)
        # convex-hull version: keep alpha in convex hull segment
        k = min(1.0, max(0.0, k))

        alpha = (1.0 - k) * alpha + k * xt

    return np.full(d, np.nan), float("nan")


In [ ]:
# Demo Kozinec on synthetic separable 2D data
N = 120
X0 = np.random.randn(2, N//2) + np.array([[-2.0], [0.0]])
X1 = np.random.randn(2, N//2) + np.array([[ 2.0], [0.0]])
X = np.hstack([X0, X1])
y = np.hstack([np.zeros(N//2, dtype=int), np.ones(N//2, dtype=int)])

w_k, b_k = kozinec(X, y, max_iterations=5000)
print("Kozinec w:", w_k)
print("Kozinec b:", b_k)

plot_binary_boundary_2d(X, y, w_k, b_k, title="Kozinec: linear boundary")


## 2) Multi-class perceptron (mean init + misclassified sample updates)

Classifier:

\[
f(x) = \arg\max_{y \in \mathcal{Y}} (w_y^T x + b_y)
\]

Training algorithm (as in the prompt):
1. Compute class means \(\mu_y\).
2. Initialize \(w_y = \mu_y\), \(b_y = 0\).
3. Repeatedly pick any misclassified sample \((x^t, y^t)\) and update:

\[
w_{y^t} \leftarrow w_{y^t} + x^t, \quad b_{y^t} \leftarrow b_{y^t} + 1
\]
\[
w_{\hat y} \leftarrow w_{\hat y} - x^t, \quad b_{\hat y} \leftarrow b_{\hat y} - 1
\]

Stop when there are no misclassified samples (zero training error), or when `max_iterations` is reached.


In [ ]:
def multiclass_perceptron(X, y, max_iterations=100000):
    """Multi-class perceptron per assignment description.

    X: (d, N)
    y: (N,) labels in {0,1,...,C-1} (or any integers)
    Returns:
      W: (C, d)
      b: (C,)
      classes: sorted unique labels (C,)
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    d, N = X.shape

    classes = np.unique(y)
    C = classes.size

    # map labels to indices 0..C-1
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    y_idx = np.array([cls_to_idx[yy] for yy in y], dtype=int)

    # 1) class means
    mus = np.zeros((C, d), dtype=float)
    for i in range(C):
        mask = (y_idx == i)
        mus[i] = X[:, mask].mean(axis=1)

    # 2) init W=means, b=0
    W = mus.copy()
    b = np.zeros(C, dtype=float)

    # 3-6) iterate
    for _ in range(int(max_iterations)):
        scores = (W @ X) + b[:, None]     # (C, N)
        y_hat = np.argmax(scores, axis=0) # (N,)

        bad = np.flatnonzero(y_hat != y_idx)
        if bad.size == 0:
            break

        t = int(bad[0])
        yt = y_idx[t]
        ypred = y_hat[t]
        xt = X[:, t]

        W[yt] += xt
        b[yt] += 1.0
        W[ypred] -= xt
        b[ypred] -= 1.0

    return W, b, classes

def multiclass_predict(X, W, b, classes):
    scores = (W @ X) + b[:, None]
    idx = np.argmax(scores, axis=0)
    return classes[idx]


In [ ]:
def plot_multiclass_boundary_2d(X, y, W, b, classes, title=None, grid=300):
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    x_min, x_max = X[0].min() - 1, X[0].max() + 1
    y_min, y_max = X[1].min() - 1, X[1].max() + 1

    xs = np.linspace(x_min, x_max, grid)
    ys = np.linspace(y_min, y_max, grid)
    XX, YY = np.meshgrid(xs, ys)
    G = np.vstack([XX.ravel(), YY.ravel()])  # (2, grid^2)

    pred = multiclass_predict(G, W, b, classes)
    # map classes to 0..C-1 for display
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    Z = np.array([cls_to_idx[p] for p in pred], dtype=int).reshape(grid, grid)

    plt.figure()
    plt.imshow(Z, origin="lower", extent=[x_min, x_max, y_min, y_max], aspect="auto", alpha=0.35)

    for c in classes:
        mask = (y == c)
        plt.scatter(X[0, mask], X[1, mask], s=18, label=f"class {c}")

    plt.legend()
    if title:
        plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()


In [ ]:
# Demo multi-class perceptron on synthetic 3-class data
N = 300
C1 = np.random.randn(2, N//3) + np.array([[-2.5], [0.0]])
C2 = np.random.randn(2, N//3) + np.array([[ 2.5], [0.0]])
C3 = np.random.randn(2, N//3) + np.array([[ 0.0], [2.5]])

Xmc = np.hstack([C1, C2, C3])
ymc = np.array([0]*(N//3) + [1]*(N//3) + [2]*(N//3), dtype=int)

Wmc, bmc, classes = multiclass_perceptron(Xmc, ymc, max_iterations=200000)
pred_train = multiclass_predict(Xmc, Wmc, bmc, classes)
train_err = np.mean(pred_train != ymc)
print(f"Multi-class training error: {train_err*100:.2f}%")

plot_multiclass_boundary_2d(Xmc, ymc, Wmc, bmc, classes, title="Multi-class perceptron decision regions")
